# 🔧 TRELLIS2 — SageMaker Full Setup & Debug
**FORGE 3D, LLC — March 2026**

### How this notebook works

| Part | When to run | What it does | Time |
|------|------------|--------------|------|
| **A** | Once (first session) | Creates S3 bucket, downloads data to EBS, builds packages, creates persistent conda env, saves pre-built wheels to S3, creates lifecycle config | ~30 min |
| **B** | Every session | Quick init — activates persistent env, verifies everything | ~30 sec |
| **C** | Debug sessions | Visual debugging pipeline (pretrained baseline, progressive swap, forensics) | ~30 min |

After Part A is done, you'll never need to re-run it. Stop/start your space freely — everything persists on EBS, and the lifecycle config handles system-level stuff automatically.

---

## PART A — ONE-TIME SETUP
⚠️ **Run this part only once.** It takes ~30 min but saves you hours on every future session.

### A1. Check GPU & Verify Instance Type

In [ ]:
import torch, os, subprocess, sys, json

if not torch.cuda.is_available():
    print('❌ NO GPU DETECTED.')
    print('You are on a CPU instance. Stop this space and restart on ml.g5.2xlarge.')
    print('See: SageMaker Studio → Spaces → your space → Stop → Edit → Instance → ml.g5.2xlarge → Run')
    raise SystemExit('GPU required')

print(f'PyTorch:   {torch.__version__}')
print(f'CUDA:      {torch.version.cuda}')
print(f'GPU:       {torch.cuda.get_device_name(0)}')
print(f'VRAM:      {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
print(f'bfloat16:  {"✅" if torch.cuda.is_bf16_supported() else "❌"}')
print(f'\nHome dir:  /home/sagemaker-user/')

# Check EBS space
stat = os.statvfs('/home/sagemaker-user')
total_gb = (stat.f_blocks * stat.f_frsize) / 1e9
free_gb = (stat.f_bavail * stat.f_frsize) / 1e9
print(f'EBS disk:  {total_gb:.0f} GB total, {free_gb:.0f} GB free')

if free_gb < 50:
    print('⚠️  Less than 50 GB free. Consider increasing EBS size in space settings.')

# Configure git identity (persists on EBS via ~/.gitconfig)
subprocess.run(['git', 'config', '--global', 'user.name', 'lobstermeeat'], check=True)
subprocess.run(['git', 'config', '--global', 'user.email', 'lobstermeeat@users.noreply.github.com'], check=True)
print(f'\nGit:       {subprocess.check_output(["git", "config", "user.name"]).decode().strip()} <{subprocess.check_output(["git", "config", "user.email"]).decode().strip()}>')

### A2. Create S3 Bucket
S3 stores your pre-built wheels so future installs take seconds instead of 15+ minutes.  
Cost: ~$0.02/month for the wheels (~500 MB).

In [ ]:
import boto3, botocore

# Get current region and account
session = boto3.session.Session()
REGION = session.region_name
ACCOUNT_ID = boto3.client('sts').get_caller_identity()['Account']

BUCKET_NAME = f'forge3d-trellis2-{ACCOUNT_ID}-{REGION}'
S3_PREFIX = 'trellis2-wheels'  # folder inside the bucket

print(f'Region:     {REGION}')
print(f'Account:    {ACCOUNT_ID}')
print(f'Bucket:     {BUCKET_NAME}')

s3 = boto3.client('s3')
try:
    s3.head_bucket(Bucket=BUCKET_NAME)
    print(f'\n✅ Bucket already exists.')
except botocore.exceptions.ClientError:
    print(f'\nCreating bucket...')
    if REGION == 'us-east-1':
        s3.create_bucket(Bucket=BUCKET_NAME)
    else:
        s3.create_bucket(
            Bucket=BUCKET_NAME,
            CreateBucketConfiguration={'LocationConstraint': REGION}
        )
    print(f'✅ Bucket created: s3://{BUCKET_NAME}/')

In [ ]:
# Save config for future sessions
config = {
    'bucket': BUCKET_NAME,
    's3_prefix': S3_PREFIX,
    'region': REGION,
    'account_id': ACCOUNT_ID,
}
config_path = os.path.expanduser('~/.trellis2_config.json')
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)
print(f'Config saved to {config_path} (persists on EBS)')

### A2b. Increase EBS Volume to 450 GB
The default 5 GB is far too small. This sets the domain default to 450 GB so all spaces get enough disk for checkpoints, conda env, and wheel cache.

In [ ]:
# Increase EBS default to 450 GB for all JupyterLab spaces in this domain
domains = sm.list_domains()['Domains']
if len(domains) >= 1:
    DOMAIN_ID = domains[0]['DomainId']
    try:
        sm.update_domain(
            DomainId=DOMAIN_ID,
            DefaultUserSettings={
                'SpaceStorageSettings': {
                    'DefaultEbsStorageSettings': {
                        'DefaultEbsVolumeSizeInGb': 450,
                        'MaximumEbsVolumeSizeInGb': 450,
                    }
                }
            }
        )
        print(f'✅ Domain {DOMAIN_ID} EBS default set to 450 GB')
    except Exception as e:
        print(f'⚠️  Could not update domain EBS settings: {e}')
        print(f'\nManual CLI command:')
        print(f'  aws --region {REGION} sagemaker update-domain \\')
        print(f'    --domain-id {DOMAIN_ID} \\')
        print(f'    --default-user-settings \'{{"SpaceStorageSettings":{{"DefaultEbsStorageSettings":{{"DefaultEbsVolumeSizeInGb":450,"MaximumEbsVolumeSizeInGb":450}}}}}}\'')
else:
    print('❌ No SageMaker domain found')

print('\n⚠️  Note: Existing spaces keep their current size.')
print('   To resize, delete and recreate the space, or use the console.')

### A3. Download Repos & Checkpoints to EBS
These only download once — they persist on your EBS volume across stop/start cycles.

In [ ]:
%%bash
set -e
cd /home/sagemaker-user

pip install -q huggingface_hub[hf_xet]

# Only download if not already present
if [ ! -d workspace ] || [ $(ls workspace/ 2>/dev/null | wc -l) -lt 3 ]; then
    echo "=== Downloading workspace ==="
    hf download lobstermeeat/trellis2-workspace \
        --local-dir workspace --repo-type model
else
    echo "✅ Workspace already on EBS, skipping download"
fi

if [ ! -d checkpoints ] || [ $(find checkpoints -name '*.pt' 2>/dev/null | wc -l) -lt 1 ]; then
    echo "=== Downloading checkpoints (final only, skipping intermediates) ==="
    hf download lobstermeeat/trellis2-checkpoints \
        --local-dir checkpoints --repo-type model \
        --include "trellis/pretrained/**" \
        --include "trellis/finetuned/*/ckpts/*step0050000*" \
        --include "hunyuan/**" \
        --include ".gitattributes"
else
    echo "✅ Checkpoints already on EBS, skipping download"
fi

if [ ! -d repos/TRELLIS.2/.git ]; then
    echo "=== Cloning TRELLIS.2 ==="
    mkdir -p repos
    git clone https://github.com/microsoft/TRELLIS.2.git repos/TRELLIS.2
else
    echo "✅ TRELLIS.2 repo already on EBS"
fi

echo ""
echo "=== EBS Contents ==="
du -sh workspace checkpoints repos 2>/dev/null || true

### A4. Create Persistent Conda Environment on EBS
System pip packages get wiped on restart. But a conda env stored in your home dir **survives**.  
This is the key trick — we install everything into this env once.

In [ ]:
%%bash
set -e

ENV_DIR="/home/sagemaker-user/.conda/envs/trellis2"

if [ -d "$ENV_DIR" ] && [ -f "$ENV_DIR/bin/python" ]; then
    echo "✅ Conda env 'trellis2' already exists on EBS"
    echo "   Python: $($ENV_DIR/bin/python --version)"
    exit 0
fi

echo "=== Creating persistent conda env ==="
conda create --yes -p $ENV_DIR python=3.12

# Register so conda can find it
conda config --add envs_dirs /home/sagemaker-user/.conda/envs

# Install ipykernel so it shows up as a Jupyter kernel
$ENV_DIR/bin/pip install ipykernel
$ENV_DIR/bin/python -m ipykernel install --user --name trellis2 --display-name "TRELLIS2 (persistent)"

echo "✅ Conda env created at $ENV_DIR"
echo "   It will persist across stop/start cycles."

### A5. Install Core Dependencies into Persistent Env
This installs everything that's available via pip (fast, no compilation).

In [ ]:
%%bash
set -e
ENV="/home/sagemaker-user/.conda/envs/trellis2"
PIP="$ENV/bin/pip"

echo "=== Installing PyTorch (GPU) ==="
$PIP install torch torchvision --index-url https://download.pytorch.org/whl/cu121 2>&1 | tail -3

echo "=== Installing Python deps ==="
$PIP install -q \
    plyfile zstandard easydict open_clip_torch safetensors \
    huggingface_hub einops scipy opencv-python-headless pandas \
    tensorboard transformers kornia timm lpips \
    scikit-learn matplotlib trimesh ipykernel Pillow 2>&1 | tail -3

echo "=== Installing utils3d ==="
$PIP install git+https://github.com/EasternJournalist/utils3d.git@9a4eb15e4021b67b12c460c7057d642626897ec8 --no-deps 2>&1 | tail -2

echo "✅ Core deps installed"
$ENV/bin/python -c "import torch; print(f'  PyTorch {torch.__version__}, CUDA {torch.version.cuda}')"

### A6. Build Native Extensions & Cache Wheels to S3
This is the slow part (~15 min): cumesh, FlexGEMM, nvdiffrast, flash-attn, xformers.  
We build them once, save the wheels to S3. Future installs just download and install — **no compilation**.

In [ ]:
%%bash
set -e
ENV="/home/sagemaker-user/.conda/envs/trellis2"
PIP="$ENV/bin/pip"
PYTHON="$ENV/bin/python"
WHEEL_DIR="/home/sagemaker-user/.wheel_cache"
mkdir -p $WHEEL_DIR

echo "=== [1/6] System deps (eigen3) ==="
sudo apt-get update -qq
sudo apt-get install -y -qq libeigen3-dev > /dev/null 2>&1
if [ -d /home/sagemaker-user/repos/TRELLIS.2/o-voxel/third_party/eigen ]; then
    cp -r /usr/include/eigen3/* /home/sagemaker-user/repos/TRELLIS.2/o-voxel/third_party/eigen/ 2>/dev/null || true
fi
echo "  Done"

echo "=== [2/6] CuMesh ★ CRITICAL ★ ==="
cd /tmp && rm -rf CuMesh
git clone --recursive https://github.com/JeffreyXiang/CuMesh.git 2>&1 | tail -1
cd CuMesh
$PIP wheel . --no-build-isolation --no-cache-dir -w $WHEEL_DIR 2>&1 | tail -3
$PIP install $WHEEL_DIR/cumesh*.whl --no-deps 2>&1 | tail -1
$PYTHON -c "import cumesh; print('  ✅ cumesh OK')"

echo "=== [3/6] FlexGEMM ==="
cd /tmp && rm -rf FlexGEMM
git clone https://github.com/JeffreyXiang/FlexGEMM.git 2>&1 | tail -1
cd FlexGEMM
$PIP wheel . --no-build-isolation --no-cache-dir -w $WHEEL_DIR 2>&1 | tail -3
$PIP install $WHEEL_DIR/flex_gemm*.whl --no-deps 2>&1 | tail -1
$PYTHON -c "import flex_gemm; print('  ✅ FlexGEMM OK')"

echo "=== [4/6] nvdiffrast ==="
cd /tmp && rm -rf nvdiffrast
git clone https://github.com/NVlabs/nvdiffrast.git 2>&1 | tail -1
cd nvdiffrast
$PIP wheel . --no-build-isolation --no-cache-dir -w $WHEEL_DIR 2>&1 | tail -3
$PIP install $WHEEL_DIR/nvdiffrast*.whl --no-deps 2>&1 | tail -1
$PYTHON -c "import nvdiffrast; print('  ✅ nvdiffrast OK')"

echo "=== [5/6] o_voxel ==="
cd /home/sagemaker-user/repos/TRELLIS.2/o-voxel
$PIP install -e . --no-build-isolation 2>&1 | tail -3
$PYTHON -c "import o_voxel; print('  ✅ o_voxel OK')"

echo "=== [6/6] flash-attn + xformers (slowest, ~10 min) ==="
mkdir -p /tmp/fa_build
MAX_JOBS=4 TMPDIR=/tmp/fa_build $PIP wheel flash-attn --no-build-isolation --no-cache-dir -w $WHEEL_DIR 2>&1 | tail -3
$PIP install $WHEEL_DIR/flash_attn*.whl 2>&1 | tail -1
$PIP install -q xformers 2>&1 | tail -2

echo ""
echo "=== Wheels saved locally ==="
ls -lhS $WHEEL_DIR/*.whl 2>/dev/null | head -10

echo ""
echo "=== Verify all imports ==="
$PYTHON -c "
import cumesh, flex_gemm, o_voxel, nvdiffrast, flash_attn, xformers
import torch, torchvision, PIL, matplotlib, sklearn
print('ALL IMPORTS OK ✅')
"

In [ ]:
# Upload wheels to S3 for fast future installs
import glob

wheel_dir = os.path.expanduser('~/.wheel_cache')
wheels = glob.glob(f'{wheel_dir}/*.whl')

print(f'Uploading {len(wheels)} wheels to s3://{BUCKET_NAME}/{S3_PREFIX}/\n')
for whl in sorted(wheels):
    name = os.path.basename(whl)
    size_mb = os.path.getsize(whl) / 1e6
    print(f'  Uploading {name} ({size_mb:.1f} MB)...')
    s3.upload_file(whl, BUCKET_NAME, f'{S3_PREFIX}/{name}')

print(f'\n✅ All wheels cached on S3.')
print(f'   Future installs: download + pip install (no compilation)')

### A7. Create Symlinks & Jupyter Kernel

In [ ]:
%%bash
# Create /workspace/ symlinks for compatibility with your original scripts
sudo mkdir -p /workspace 2>/dev/null || true

for link_target in \
    "repos:/home/sagemaker-user/repos" \
    "checkpoints:/home/sagemaker-user/checkpoints" \
    "data:/home/sagemaker-user/data" \
    "configs:/home/sagemaker-user/workspace/configs"; do
    link="/workspace/${link_target%%:*}"
    target="${link_target#*:}"
    if [ -d "$target" ] && [ ! -e "$link" ]; then
        sudo ln -sf "$target" "$link"
        echo "🔗 $link → $target"
    fi
done

# Re-register Jupyter kernel (in case it was lost)
ENV="/home/sagemaker-user/.conda/envs/trellis2"
$ENV/bin/python -m ipykernel install --user --name trellis2 --display-name "TRELLIS2 (persistent)" 2>&1 | tail -1

echo "✅ Symlinks and kernel configured"

### A8. Create Lifecycle Configuration Script
This script runs **automatically every time your space starts**.  
It handles the non-persistent stuff: system packages, symlinks, re-activating the env.

⚠️ Lifecycle configs have a **5-minute limit**. That's why we pre-built the wheels — this script just downloads and installs them, no compilation.

In [ ]:
# Write the lifecycle script
lcc_script = f"""#!/bin/bash
set -eux

# ─── Lifecycle Config: TRELLIS2 Auto-Setup ───
# Runs automatically on every space start.
# Pre-built wheels on S3 → no compilation needed → finishes in ~2 min.

HOME_DIR=/home/sagemaker-user
ENV_DIR=$HOME_DIR/.conda/envs/trellis2
WHEEL_CACHE=$HOME_DIR/.wheel_cache
BUCKET="{BUCKET_NAME}"
PREFIX="{S3_PREFIX}"

# [1] System deps (fast, ~10 sec)
sudo apt-get update -qq
sudo apt-get install -y -qq libeigen3-dev > /dev/null 2>&1 || true
if [ -d $HOME_DIR/repos/TRELLIS.2/o-voxel/third_party/eigen ]; then
    cp -r /usr/include/eigen3/* $HOME_DIR/repos/TRELLIS.2/o-voxel/third_party/eigen/ 2>/dev/null || true
fi

# [2] Create /workspace symlinks (wiped on restart)
sudo mkdir -p /workspace 2>/dev/null || true
for pair in "repos:$HOME_DIR/repos" "checkpoints:$HOME_DIR/checkpoints" "data:$HOME_DIR/data" "configs:$HOME_DIR/workspace/configs"; do
    link="/workspace/${{pair%%:*}}"
    target="${{pair#*:}}"
    [ -d "$target" ] && [ ! -e "$link" ] && sudo ln -sf "$target" "$link" || true
done

# [3] Check if conda env exists on EBS
if [ ! -f "$ENV_DIR/bin/python" ]; then
    echo "⚠️ Conda env not found on EBS. Run Part A of the setup notebook first."
    exit 0
fi

# [4] Re-install compiled wheels if they got wiped
# (The conda env is on EBS so packages usually persist,
#  but if the env gets corrupted we can restore from S3)
$ENV_DIR/bin/python -c "import cumesh" 2>/dev/null || {{
    echo "Restoring wheels from S3..."
    mkdir -p $WHEEL_CACHE
    aws s3 sync s3://$BUCKET/$PREFIX/ $WHEEL_CACHE/ --quiet
    $ENV_DIR/bin/pip install $WHEEL_CACHE/*.whl --no-deps --quiet 2>&1 | tail -3
}}

# [5] Re-install o_voxel in editable mode (lives in the repo on EBS)
if [ -d $HOME_DIR/repos/TRELLIS.2/o-voxel ]; then
    $ENV_DIR/bin/pip install -e $HOME_DIR/repos/TRELLIS.2/o-voxel --no-build-isolation --quiet 2>&1 | tail -1 || true
fi

# [6] Register Jupyter kernel
$ENV_DIR/bin/python -m ipykernel install --user --name trellis2 --display-name "TRELLIS2 (persistent)" 2>/dev/null || true

echo "✅ TRELLIS2 lifecycle config complete"
"""

lcc_path = os.path.expanduser('~/trellis2_lifecycle.sh')
with open(lcc_path, 'w') as f:
    f.write(lcc_script)
os.chmod(lcc_path, 0o755)

print(f'Lifecycle script saved to {lcc_path}')
print(f'Script size: {len(lcc_script)} chars (limit: 16384)')

In [ ]:
# Register the lifecycle config with SageMaker
import base64

sm = boto3.client('sagemaker')

with open(lcc_path, 'rb') as f:
    lcc_content = base64.b64encode(f.read()).decode('utf-8')

LCC_NAME = 'trellis2-auto-setup'

# Create or update the lifecycle config
try:
    response = sm.create_studio_lifecycle_config(
        StudioLifecycleConfigName=LCC_NAME,
        StudioLifecycleConfigContent=lcc_content,
        StudioLifecycleConfigAppType='JupyterLab'
    )
    LCC_ARN = response['StudioLifecycleConfigArn']
    print(f'✅ Created lifecycle config: {LCC_NAME}')
    print(f'   ARN: {LCC_ARN}')
except sm.exceptions.ResourceInUse:
    # Already exists — get the ARN
    response = sm.describe_studio_lifecycle_config(
        StudioLifecycleConfigName=LCC_NAME
    )
    LCC_ARN = response['StudioLifecycleConfigArn']
    print(f'✅ Lifecycle config already exists: {LCC_NAME}')
    print(f'   ARN: {LCC_ARN}')
    print(f'   To update it, delete first then re-create:')
    print(f'   sm.delete_studio_lifecycle_config(StudioLifecycleConfigName="{LCC_NAME}")')

In [ ]:
# Attach the lifecycle config to your domain
# This makes it run automatically when any JupyterLab space starts

# Find your domain ID
domains = sm.list_domains()['Domains']
if len(domains) == 1:
    DOMAIN_ID = domains[0]['DomainId']
    print(f'Domain: {DOMAIN_ID}')
elif len(domains) > 1:
    print('Multiple domains found. Pick yours:')
    for d in domains:
        print(f'  {d["DomainId"]}: {d["DomainName"]}')
    DOMAIN_ID = domains[0]['DomainId']  # ← change index if needed
else:
    print('❌ No SageMaker domain found')
    DOMAIN_ID = None

if DOMAIN_ID:
    try:
        sm.update_domain(
            DomainId=DOMAIN_ID,
            DefaultUserSettings={
                'JupyterLabAppSettings': {
                    'LifecycleConfigArns': [LCC_ARN],
                }
            }
        )
        print(f'\n✅ Lifecycle config attached to domain.')
        print(f'   It will run automatically on every space start.')
    except Exception as e:
        print(f'\n⚠️  Could not auto-attach: {e}')
        print(f'\nManual steps:')
        print(f'  1. Go to SageMaker Console → Domains → {DOMAIN_ID}')
        print(f'  2. Click "Environment" tab')
        print(f'  3. Under "Lifecycle configurations", click "Attach"')
        print(f'  4. Select "{LCC_NAME}" and click "Set as default"')

### A9. Verify Persistence
Let's check that everything that should persist is in the right place.

In [ ]:
# Final status report
print('═' * 60)
print('  SETUP STATUS REPORT')
print('═' * 60)

checks = [
    ('S3 bucket',          f's3://{BUCKET_NAME}/', True),
    ('Config file (EBS)',   '~/.trellis2_config.json', os.path.exists(os.path.expanduser('~/.trellis2_config.json'))),
    ('Conda env (EBS)',    '~/.conda/envs/trellis2/', os.path.exists(os.path.expanduser('~/.conda/envs/trellis2/bin/python'))),
    ('Wheel cache (EBS)',  '~/.wheel_cache/', len(glob.glob(os.path.expanduser('~/.wheel_cache/*.whl'))) > 0),
    ('TRELLIS repo (EBS)', '~/repos/TRELLIS.2/', os.path.isdir(os.path.expanduser('~/repos/TRELLIS.2'))),
    ('Checkpoints (EBS)',  '~/checkpoints/', os.path.isdir(os.path.expanduser('~/checkpoints'))),
    ('Workspace (EBS)',    '~/workspace/', os.path.isdir(os.path.expanduser('~/workspace'))),
    ('Lifecycle config',   LCC_NAME, True),
    ('Lifecycle script',   '~/trellis2_lifecycle.sh', os.path.exists(os.path.expanduser('~/trellis2_lifecycle.sh'))),
]

all_ok = True
for name, location, ok in checks:
    status = '✅' if ok else '❌'
    if not ok: all_ok = False
    print(f'  {status} {name:25s} {location}')

print()
if all_ok:
    print('🎉 SETUP COMPLETE!')
    print()
    print('What persists across stop/start (on EBS):')
    print('  ✅ Conda env with all packages')
    print('  ✅ Model checkpoints')
    print('  ✅ TRELLIS.2 source code')
    print('  ✅ Pre-built wheels')
    print('  ✅ This notebook')
    print()
    print('What the lifecycle config restores on each start:')
    print('  🔄 System packages (libeigen3-dev)')
    print('  🔄 /workspace/ symlinks')
    print('  🔄 Jupyter kernel registration')
    print('  🔄 Compiled wheels (if env got corrupted)')
    print()
    print('Next time you start this space, skip to PART B.')
else:
    print('⚠️  Some checks failed. Review the output above.')

---
## PART B — SESSION INIT (run every time)

⚠️ **Switch kernel first!** At the top-right of this notebook, click the kernel name and select **"TRELLIS2 (persistent)"**. This uses the conda env you created in Part A.

If you don't see it, run in a terminal:
```
/home/sagemaker-user/.conda/envs/trellis2/bin/python -m ipykernel install --user --name trellis2 --display-name "TRELLIS2 (persistent)"
```
Then refresh the page.

In [ ]:
# ══════════════════════════════════════════════════════════
#  SESSION INIT — ~30 seconds
# ══════════════════════════════════════════════════════════
import os, sys, json, torch, importlib
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Verify we're in the right kernel
env_python = sys.executable
if '.conda/envs/trellis2' not in env_python:
    print(f'⚠️  Wrong kernel! Current: {env_python}')
    print('Switch to "TRELLIS2 (persistent)" kernel using the selector at top-right.')
    print('If it\'s not listed, the lifecycle config or Part A may not have run yet.')
else:
    print(f'✅ Using persistent env: {env_python}')

os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['ATTN_BACKEND'] = 'xformers'

HOME = '/home/sagemaker-user'
sys.path.insert(0, f'{HOME}/repos/TRELLIS.2')

from trellis2.pipelines import Trellis2ImageTo3DPipeline
import o_voxel

# Load config
config_path = os.path.expanduser('~/.trellis2_config.json')
if os.path.exists(config_path):
    with open(config_path) as f:
        cfg = json.load(f)
    BUCKET_NAME = cfg['bucket']
    S3_PREFIX = cfg['s3_prefix']
else:
    BUCKET_NAME = S3_PREFIX = None
    print('⚠️  No config file found. Run Part A first.')

# Paths
PRETRAINED_DIR = f'{HOME}/checkpoints/trellis/pretrained'
FINETUNED_SS   = f'{HOME}/checkpoints/trellis/finetuned/ss_flow/ckpts/denoiser_step0050000.pt'
FINETUNED_SH   = f'{HOME}/checkpoints/trellis/finetuned/slat_flow_shape/ckpts/denoiser_step0050000.pt'
FINETUNED_TEX  = f'{HOME}/checkpoints/trellis/finetuned/slat_flow_tex/ckpts/denoiser_step0050000.pt'
EXAMPLE_IMG    = f'{HOME}/repos/TRELLIS.2/assets/example_image/T.png'
DEBUG_DIR      = f'{HOME}/debug_output'
os.makedirs(DEBUG_DIR, exist_ok=True)

# Quick import check
for name in ['cumesh', 'flex_gemm', 'o_voxel', 'nvdiffrast']:
    try:
        __import__(name)
        print(f'  {name}: ✅')
    except:
        print(f'  {name}: ❌ (run Part A or check lifecycle config logs)')

print(f'\nGPU: {torch.cuda.get_device_name(0)}')
print(f'Debug output → {DEBUG_DIR}/')
print('\nSession ready ✅')

In [ ]:
# ══════════════════════════════════════════════════════════
#  HELPERS
# ══════════════════════════════════════════════════════════

def save_fig(name):
    plt.savefig(os.path.join(DEBUG_DIR, name), dpi=150, bbox_inches='tight')

def render_mesh(verts, faces, title='Mesh', filename=None, n_views=6):
    v = verts.cpu().numpy() if torch.is_tensor(verts) else np.asarray(verts)
    f = faces.cpu().numpy() if torch.is_tensor(faces) else np.asarray(faces)
    v = (v - (v.max(0)+v.min(0))/2) / ((v.max(0)-v.min(0)).max()+1e-8)
    if len(f) > 200000: f = f[np.random.choice(len(f), 200000, replace=False)]
    cols = min(n_views, 3)
    rows = max(1, (n_views+cols-1)//cols)
    fig, axes = plt.subplots(rows, cols, figsize=(6*cols, 6*rows), subplot_kw={'projection':'3d'})
    for i, ax in enumerate(np.array(axes).flat):
        if i >= n_views: ax.set_visible(False); continue
        ax.plot_trisurf(v[:,0],v[:,1],f,v[:,2],color='#4A90D9',edgecolor='#2E75B6',linewidth=.05,alpha=.8)
        ax.view_init(30, i*360//n_views); ax.set_axis_off()
        ax.set_xlim(-.6,.6); ax.set_ylim(-.6,.6); ax.set_zlim(-.6,.6)
    plt.suptitle(f'{title} ({len(v)} verts)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    if filename: save_fig(filename)
    plt.show()

def compare_meshes(mesh_list, labels, title='', filename=None):
    n = len(mesh_list)
    fig, axes = plt.subplots(1, n, figsize=(7*n, 7), subplot_kw={'projection':'3d'})
    if n == 1: axes = [axes]
    for ax, (verts, faces), label in zip(axes, mesh_list, labels):
        v = verts.cpu().numpy() if torch.is_tensor(verts) else np.asarray(verts)
        f = faces.cpu().numpy() if torch.is_tensor(faces) else np.asarray(faces)
        v = (v - (v.max(0)+v.min(0))/2) / ((v.max(0)-v.min(0)).max()+1e-8)
        if len(f) > 200000: f = f[np.random.choice(len(f), 200000, replace=False)]
        ax.plot_trisurf(v[:,0],v[:,1],f,v[:,2],color='#4A90D9',edgecolor='#2E75B6',linewidth=.05,alpha=.8)
        ax.view_init(30, 45); ax.set_axis_off()
        ax.set_title(f'{label}\n({len(v)} verts)', fontsize=11)
    plt.suptitle(title, fontsize=16, fontweight='bold')
    plt.tight_layout()
    if filename: save_fig(filename)
    plt.show()

def run_pipeline(overrides=None, seed=42):
    pipe = Trellis2ImageTo3DPipeline.from_pretrained(PRETRAINED_DIR)
    if overrides:
        for key, path in overrides.items():
            if os.path.exists(path):
                ckpt = torch.load(path, map_location='cpu')
                r = pipe.models[key].load_state_dict(ckpt, strict=False)
                print(f'  Loaded {key} (missing={len(r.missing_keys)}, unexpected={len(r.unexpected_keys)})')
            else:
                print(f'  ❌ Not found: {path}')
    pipe.cuda()
    mesh = pipe.run(Image.open(EXAMPLE_IMG), seed=seed)[0]
    mesh.simplify(16777216)
    del pipe; torch.cuda.empty_cache()
    return mesh

print('Helpers ready ✅')

---
## PART C — VISUAL DEBUGGING

### C1. Identify Pipeline Model Keys

In [ ]:
pipe_tmp = Trellis2ImageTo3DPipeline.from_pretrained(PRETRAINED_DIR)
print('Pipeline model keys:')
for k, v in pipe_tmp.models.items():
    n = sum(p.numel() for p in v.parameters()) / 1e6
    print(f'  "{k}"  →  {type(v).__name__}  ({n:.0f}M params)')
del pipe_tmp; torch.cuda.empty_cache()

# ⚠️ UPDATE THESE if the keys above differ from what's below:
SS_KEY    = 'sparse_structure_flow_model'
SHAPE_KEY = 'shape_slat_flow_model_512'
TEX_KEY   = 'tex_slat_flow_model_512'

### C2. Phase 1 — Pretrained Baseline

In [ ]:
image = Image.open(EXAMPLE_IMG)
plt.figure(figsize=(4,4)); plt.imshow(image); plt.axis('off')
plt.title(f'Input ({image.size[0]}×{image.size[1]})'); save_fig('p1_input.png'); plt.show()

print('Running pretrained...')
mesh_pre = run_pipeline()
print(f'Vertices: {mesh_pre.vertices.shape}, Faces: {mesh_pre.faces.shape}')

render_mesh(mesh_pre.vertices, mesh_pre.faces,
            title='Phase 1: PRETRAINED Baseline', filename='p1_pretrained.png')
# ✅ Recognizable shape  |  ❌ Blob → environment broken, stop here

### C3. Phase 2 — Progressive Weight Swap ★ THE DEFINITIVE TEST ★
Swap each finetuned checkpoint one at a time. Whichever introduces blobs = broken.

In [ ]:
configs = [
    ('All Pretrained',     {}),
    ('+SS Finetuned',      {SS_KEY: FINETUNED_SS}),
    ('+Shape Finetuned',   {SHAPE_KEY: FINETUNED_SH}),
    ('+Texture Finetuned', {TEX_KEY: FINETUNED_TEX}),
]
swap_results = []
for name, overrides in configs:
    print(f'\n{"═"*50}\n  {name}\n{"═"*50}')
    mesh = run_pipeline(overrides)
    swap_results.append((name, mesh))
    print(f'  → {mesh.vertices.shape[0]} verts')
print('\n✅ All 4 runs complete.')

In [ ]:
# 🖼️ THE MONEY SHOT
compare_meshes(
    [(m.vertices, m.faces) for _, m in swap_results],
    [n for n, _ in swap_results],
    title='Which Checkpoint Breaks It?',
    filename='p2_swap.png'
)
# Most likely: Panel 3 (+Shape Finetuned) = blob

### C4. Phase 3 — Checkpoint Forensics

In [ ]:
# ⚠️ Set to whichever checkpoint was the blob in Phase 2
BAD_CKPT = FINETUNED_SH
BAD_KEY  = SHAPE_KEY

pipe_ref = Trellis2ImageTo3DPipeline.from_pretrained(PRETRAINED_DIR)
pre_sd = {k: v.float().cpu() for k, v in pipe_ref.models[BAD_KEY].state_dict().items()}
ft_sd = {k: v.float().cpu() for k, v in torch.load(BAD_CKPT, map_location='cpu').items()}
del pipe_ref; torch.cuda.empty_cache()

shared = sorted(set(pre_sd) & set(ft_sd))
identical = nan_ct = 0
max_diff = 0
diffs = []
for key in shared:
    pw, fw = pre_sd[key], ft_sd[key]
    if pw.shape != fw.shape: continue
    ident = torch.allclose(pw, fw, atol=1e-6)
    has_nan = fw.isnan().any().item()
    diff = (pw - fw).abs().max().item()
    if ident: identical += 1
    if has_nan: nan_ct += 1
    max_diff = max(max_diff, diff)
    diffs.append((key, diff, ident, has_nan))

print(f'{"═"*60}\n  FORENSICS: {os.path.basename(BAD_CKPT)}\n{"═"*60}')
print(f'Identical to pretrained: {identical}/{len(shared)} ({100*identical/max(len(shared),1):.1f}%)')
print(f'Contains NaN:           {nan_ct} layers')
print(f'Max abs difference:     {max_diff:.6f}')

if identical == len(shared):
    print('\n❌ VERDICT: IDENTICAL to pretrained. Training never updated weights.')
    print('   → Retrain with fixed cumesh.')
elif nan_ct > 0:
    print(f'\n❌ VERDICT: {nan_ct} layers NaN. Training diverged.')
elif max_diff < 0.001:
    print('\n⚠️  VERDICT: Barely changed. Very few steps ran.')
else:
    print(f'\n✅ VERDICT: Weights differ (max diff: {max_diff:.4f}). Training ran.')

In [ ]:
# 🖼️ Weight drift heatmap
fig, ax = plt.subplots(figsize=(20, 5))
vals = [d[1] for d in diffs]
colors = ['#E74C3C' if d[3] else '#3498DB' for d in diffs]
ax.bar(range(len(vals)), vals, color=colors, alpha=.7, width=1.)
ax.set_xlabel('Layer Index'); ax.set_ylabel('Max |Pretrained − Finetuned|')
ax.set_title(f'Weight Drift: {os.path.basename(BAD_CKPT)}', fontsize=14, fontweight='bold')
ax.axhline(y=.001, color='orange', ls='--', alpha=.7, label='Barely-changed')
ax.legend()
for i, d in enumerate(diffs):
    if d[3]: ax.annotate('NaN', (i, d[1]), fontsize=8, color='red', fontweight='bold', ha='center')
plt.tight_layout(); save_fig('p3_drift.png'); plt.show()

### C5. Export All Debug Artifacts

In [ ]:
print(f'Debug artifacts in {DEBUG_DIR}/:\n')
total = 0
for f in sorted(os.listdir(DEBUG_DIR)):
    size = os.path.getsize(os.path.join(DEBUG_DIR, f))
    total += size
    print(f'  {f:40s} {size/1024:.0f} KB')
print(f'\nTotal: {total/1e6:.1f} MB')

---
## Decision Tree

| Finding | Action |
|---------|--------|
| Phase 1 pretrained = blob | Environment broken. Rebuild env (re-run Part A). |
| Phase 1 pretrained = good | Continue to Phase 2. |
| Phase 2 +Shape = blob | SLAT shape checkpoint broken. Retrain with fixed cumesh. |
| Phase 2 +SS = blob | SS finetuning broke. Use pretrained SS weights. |
| Phase 2 +Tex = blob | Texture finetuning broke. Use pretrained tex weights. |
| Phase 3 identical to pretrained | Training saved before crash. Retrain. |
| Phase 3 NaN | Training diverged. Lower lr, check data. |
| Phase 3 differs but blob | Overfitting or data issue. Fewer steps / more data. |

**Most likely:** Phase 1 ✅ → Phase 2 confirms Shape blob → Phase 3 shows identical → **Retrain SLAT shape.**